#### SENTIMENT CLASSIFICATION USING AUTOENCODER

In [ ]:
"""
### PART 1: Sentiment Classification using Autoencoder (TensorFlow / Keras)

#### 1. Objective and Concept
The idea behind this experiment is unsupervised feature reduction. Instead of using complex sequence models like RNNs or Transformers, the goal is to pass high-dimensional text sequences through an autoencoder to compress the text into a compact 32-dimensional "latent bottleneck." Then, a small classification network is trained on these compressed features to predict whether a movie review is positive or negative.

#### 2. Data Preparation
- **Dataset**: The IMDB Movie Reviews dataset (`imdb.load_data`), containing 25,000 training reviews and 25,000 testing reviews labeled 0 (negative) or 1 (positive).
- **Vocabulary Filtering**: The vocabulary is restricted to the top 10,000 most frequent words.
- **Sequence Standardization**: All reviews are forced to an identical length of 200 words using post-padding (shorter reviews are padded with zeros at the end; longer reviews are truncated to 200 words).
- **Normalization**: The integer word IDs are converted to floating-point values and divided by the vocabulary size (10,000) so input values fall between 0 and 1.

#### 3. Model Architecture and Pipeline
The pipeline operates in two sequential stages:

- **Stage 1 — The Deep Autoencoder**:
  - **Encoder**: Compresses the 200-dimensional review input into progressively smaller hidden representations: 200 inputs $\rightarrow$ 128 neurons $\rightarrow$ 64 neurons $\rightarrow$ 32 neurons (the bottleneck latent layer). All hidden layers use ReLU activation.
  - **Decoder**: Attempts to reconstruct the original 200-dimensional sequence from the 32 latent features: 32 latent neurons $\rightarrow$ 64 neurons $\rightarrow$ 128 neurons $\rightarrow$ 200 output neurons with Sigmoid activation.
  - **Autoencoder Training**: Trained for 5 epochs using the Adam optimizer and Mean Squared Error (MSE) loss, with a batch size of 128 and an 80/20 train/validation split.

- **Stage 2 — Feature Extraction and Classification**:
  - The trained decoder is discarded. The encoder is used to transform both the training and test reviews into 32-dimensional feature vectors.
  - A small downstream classifier is built: 32 input features $\rightarrow$ 16 hidden neurons (ReLU) $\rightarrow$ 1 output neuron (Sigmoid).
  - The classifier is trained for 5 epochs using Binary Cross-Entropy loss.

#### 4. Existing Training Outputs and Results
- **Autoencoder Reconstruction Loss**:
  - In Epoch 1, the reconstruction loss started at 0.0451 (validation loss 0.0243).
  - Across epochs 2 through 5, the loss leveled off at 0.0241 (validation loss 0.0242).
- **Classifier Training**:
  - In Epoch 1, classification accuracy started at 49.86% (loss 0.6972).
  - Across epochs 2 to 5, accuracy remained nearly flat around 50.3% to 51.25%, with the loss plateauing around 0.6923.
- **Final Test Evaluation (25,000 Test Reviews)**:
  - **Overall Accuracy**: **50.56%** (essentially equivalent to a random 50/50 coin toss).
  - **Negative Reviews**: Precision of 0.51, Recall of 0.44, and an F1-score of 0.47 across 12,500 samples.
  - **Positive Reviews**: Precision of 0.50, Recall of 0.57, and an F1-score of 0.54 across 12,500 samples.

#### 5. Why Did the Classifier Achieve ~50% Accuracy?
This experiment highlights a fundamental concept in NLP:
- Treating word index numbers as continuous numerical scalars (e.g., word #9000 is treated as 9 times larger than word #1000) destroys natural semantic meaning.
- Without an embedding layer or recurrent mechanism, a dense autoencoder primarily memorizes the position of the trailing zeros (padding) rather than learning sentiment signals, illustrating why dense autoencoders are poorly suited for raw token sequences compared to embeddings or word-frequency bags (TF-IDF).

---

### PART 2: Variational Autoencoder (VAE) on MNIST (PyTorch)

#### 1. Objective and Concept
Unlike standard autoencoders that simply compress data into arbitrary points, a **Variational Autoencoder (VAE)** is a probabilistic generative model. It maps input data into a smooth, continuous probability distribution (mean and variance). This ensures that any point chosen in the latent space can be decoded into a realistic new image, allowing synthetic data generation.

#### 2. Data Preparation
- **Dataset**: MNIST handwritten digits (0 through 9).
- **Image Transformation**: Each $28 \times 28$ grayscale image is normalized to the $[0, 1]$ interval and flattened into a 784-dimensional vector.
- **Batching**: Loaded with a batch size of 128 into PyTorch DataLoaders.

#### 3. VAE Architecture and the Reparameterization Trick
- **Encoder**:
  - Takes 784 pixel inputs $\rightarrow$ Fully Connected Layer (256 neurons, ReLU) $\rightarrow$ Fully Connected Layer (64 neurons, ReLU).
  - The 64 features branch into two parallel linear heads:
    - **Mean vector ($\mu$)**: 2 dimensions.
    - **Log-variance vector ($\log\sigma^2$)**: 2 dimensions.
- **The Reparameterization Trick**:
  - Instead of sampling directly from $\mathcal{N}(\mu, \sigma^2)$ (which blocks backpropagation because random sampling cannot compute gradients), the model samples an auxiliary noise vector $\epsilon$ from a standard normal distribution $\mathcal{N}(0, I)$ and computes $z = \mu + \sigma \cdot \epsilon$. This allows gradients to flow backwards through $\mu$ and $\sigma$.
- **Decoder**:
  - Takes the 2-dimensional sampled latent point $z \rightarrow$ Fully Connected Layer (64 neurons, ReLU) $\rightarrow$ Fully Connected Layer (256 neurons, ReLU) $\rightarrow$ Output Layer (784 neurons, Sigmoid activation) to reconstruct the digit image.

#### 4. The Loss Function
The model optimizes two competing objectives simultaneously:
1. **Reconstruction Loss (Binary Cross-Entropy)**: Measures how accurately the reconstructed image matches the original digit pixels.
2. **Kullback-Leibler (KL) Divergence**: Measures how closely the latent space distribution matches a standard normal distribution $\mathcal{N}(0, I)$. This prevents the encoder from clustering digits too far apart and creates a continuous latent space.

#### 5. Training Progression Across 15 Epochs
The model was trained for 15 epochs using the Adam optimizer with a learning rate of 0.001:
- **Epoch 1**: Total Loss was 192.47 (Reconstruction Loss: 188.51, KL Loss: 3.96).
- **Epoch 2**: Total Loss dropped sharply to 166.12 (Reconstruction Loss: 161.18, KL Loss: 4.94).
- **Epochs 3 to 7**: Total Loss steadily declined from 161.07 down to 150.65 as the reconstructions sharpened and digits became clearer.
- **Epochs 8 to 12**: Total Loss dropped further from 149.32 down to 145.83, while KL divergence leveled off around 6.22, showing the latent distribution had stabilized.
- **Epoch 15 (Final)**: Total Loss settled at **144.20** (Reconstruction Loss: **137.87**, KL Loss: **6.33**).

#### 6. Generated Visualizations and Outputs
The PyTorch code produced two visual artifacts saved to the output directory:
- **`vae_pytorch_generated_digits.png`**:
  A 15-by-15 grid (225 synthetic images) generated by scanning the 2D latent space coordinates from -3 to +3 and feeding each point through the decoder. It showcases smooth transitions where digits smoothly morph from one digit to another (e.g., from a 0 gradually transforming into a 6, then into an 8, then a 3).
- **`vae_pytorch_latent_space.png`**:
  A 2D scatter plot projecting all training images into the two latent dimensions $(z_0, z_1)$, color-coded by their actual digit class (0 through 9). It visually demonstrates that the VAE successfully grouped similar digits into contiguous clusters in continuous 2D space.
  """
#Name: Kallur Yathin
#Section: IT-A
#Roll number: 3122245002036
# Import necessary libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.metrics import accuracy_score, classification_report

# Load IMDB dataset
vocab_size = 10000
max_len = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

# Make all reviews same length
x_train = pad_sequences(x_train, maxlen=max_len, padding='post')
x_test = pad_sequences(x_test, maxlen=max_len, padding='post')

# Normalize
x_train = x_train.astype('float32') / vocab_size
x_test = x_test.astype('float32') / vocab_size

# -------- Autoencoder --------
input_layer = Input(shape=(max_len,))

# Encoder
encoded = Dense(128, activation='relu')(input_layer)
encoded = Dense(64, activation='relu')(encoded)
latent = Dense(32, activation='relu', name='latent_layer')(encoded)

# Decoder
decoded = Dense(64, activation='relu')(latent)
decoded = Dense(128, activation='relu')(decoded)
decoded = Dense(max_len, activation='sigmoid')(decoded)

autoencoder = Model(input_layer, decoded)

autoencoder.compile(optimizer='adam', loss='mse')

# Train Autoencoder
autoencoder.fit(
    x_train, x_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

# -------- Extract the latent features --------
encoder = Model(input_layer, latent)

X_train_features = encoder.predict(x_train, verbose=0)
X_test_features = encoder.predict(x_test, verbose=0)

# -------- Sentiment Classifier --------
classifier_input = Input(shape=(32,))
x = Dense(16, activation='relu')(classifier_input)
output = Dense(1, activation='sigmoid')(x)

classifier = Model(classifier_input, output)

classifier.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

classifier.fit(
    X_train_features, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

# -------- Prediction --------
y_prob = classifier.predict(X_test_features, verbose=0)
y_pred = (y_prob > 0.5).astype(int).ravel()

# -------- Results --------
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Negative', 'Positive']
))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0451 - val_loss: 0.0243
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0242 - val_loss: 0.0243
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0242 - val_loss: 0.0243
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0241 - val_loss: 0.0242
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0241 - val_loss: 0.0242
Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4986 - loss: 0.6972 - val_accuracy: 0.4938 - val_loss: 0.6996
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5030 - loss: 0.6939 - val_accuracy: 0.5106 - val_loss: 0.6928
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5063 - loss: 0.6928 - val_accuracy: 0.5002 - val_loss: 0.6922
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5125 - loss: 0.6927 - val_accuracy: 0.5228 - val_loss: 0.6920
Epoch 5/5
157/15

### VARIATIONAL AUTOENCODER (VAE) ON MNIST -- PYTORCH VERSION

In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os # Import the os module

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Creating output directory if it doesn't exist
output_dir = "/mnt/user-data/outputs/"
os.makedirs(output_dir, exist_ok=True)


# 1. Load MNIST Dataset

original_dim = 784
latent_dim = 2   # 2D so we can easily visualize/generate & plot the latent space
batch_size = 128

transform = transforms.Compose([
    transforms.ToTensor(),                       # scales pixels to [0, 1]
    transforms.Lambda(lambda x: x.view(-1)),      # flatten 28x28 -> 784
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


# VAE model: encoder -> reparameterize -> decoder

class VAE(nn.Module):
    def __init__(self, original_dim=784, latent_dim=2):
        super().__init__()
        # Encoder
        self.enc_fc1 = nn.Linear(original_dim, 256)
        self.enc_fc2 = nn.Linear(256, 64)
        self.fc_mean = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)

        # Decoder
        self.dec_fc1 = nn.Linear(latent_dim, 64)
        self.dec_fc2 = nn.Linear(64, 256)
        self.dec_out = nn.Linear(256, original_dim)

    def encode(self, x):
        h = F.relu(self.enc_fc1(x))
        h = F.relu(self.enc_fc2(h))
        return self.fc_mean(h), self.fc_logvar(h)

    def reparameterize(self, z_mean, z_logvar):
        # The "reparameterization trick": sample z = mean + std * epsilon
        # so gradients can flow through the sampling step.
        std = torch.exp(0.5 * z_logvar)
        epsilon = torch.randn_like(std)
        return z_mean + std * epsilon

    def decode(self, z):
        h = F.relu(self.dec_fc1(z))
        h = F.relu(self.dec_fc2(h))
        return torch.sigmoid(self.dec_out(h))

    def forward(self, x):
        z_mean, z_logvar = self.encode(x)
        z = self.reparameterize(z_mean, z_logvar)
        reconstruction = self.decode(z)
        return reconstruction, z_mean, z_logvar


# Loss = reconstruction loss + KL divergence

def vae_loss(reconstruction, x, z_mean, z_logvar):
    # Reconstruction loss: binary cross-entropy summed over pixels
    recon_loss = F.binary_cross_entropy(reconstruction, x, reduction="sum")
    # KL divergence: keeps the latent distribution close to N(0, 1)
    kl_loss = -0.5 * torch.sum(1 + z_logvar - z_mean.pow(2) - z_logvar.exp())
    return recon_loss + kl_loss, recon_loss, kl_loss


# Train the model

model = VAE(original_dim, latent_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 15

for epoch in range(1, epochs + 1):
    model.train()
    total_loss, total_recon, total_kl, n_samples = 0.0, 0.0, 0.0, 0

    for x, _ in train_loader:
        x = x.to(device)
        optimizer.zero_grad()

        reconstruction, z_mean, z_logvar = model(x)
        loss, recon_loss, kl_loss = vae_loss(reconstruction, x, z_mean, z_logvar)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
        n_samples += x.size(0)

    print(
        f"Epoch {epoch:2d}/{epochs} | "
        f"loss: {total_loss / n_samples:.4f} | "
        f"recon: {total_recon / n_samples:.4f} | "
        f"kl: {total_kl / n_samples:.4f}"
    )


# Generate new digits by sampling a grid over latent space

model.eval()
n = 15
digit_size = 28
figure = np.zeros((digit_size * n, digit_size * n))

grid_x = np.linspace(-3, 3, n)
grid_y = np.linspace(-3, 3, n)[::-1]

with torch.no_grad():
    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z_sample = torch.tensor([[xi, yi]], dtype=torch.float32).to(device)
            x_decoded = model.decode(z_sample).cpu().numpy()
            digit = x_decoded[0].reshape(digit_size, digit_size)
            figure[
                i * digit_size : (i + 1) * digit_size,
                j * digit_size : (j + 1) * digit_size,
            ] = digit

plt.figure(figsize=(10, 10))
plt.imshow(figure, cmap="Greys_r")
plt.axis("off")
plt.title("Digits generated by sampling the VAE latent space")
plt.savefig(os.path.join(output_dir, "vae_pytorch_generated_digits.png"), dpi=150, bbox_inches="tight")
plt.close()


# Visualize how real digits cluster in latent space (colored by true label)

all_z_mean = []
all_labels = []

with torch.no_grad():
    for x, y in train_loader:
        x = x.to(device)
        z_mean, _ = model.encode(x)
        all_z_mean.append(z_mean.cpu().numpy())
        all_labels.append(y.numpy())

all_z_mean = np.concatenate(all_z_mean, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(all_z_mean[:, 0], all_z_mean[:, 1], c=all_labels, cmap="tab10", s=2)
plt.colorbar(scatter, label="Digit label")
plt.xlabel("z[0]")
plt.ylabel("z[1]")
plt.title("MNIST digits mapped into 2D latent space")
plt.savefig(os.path.join(output_dir, "vae_pytorch_latent_space.png"), dpi=150, bbox_inches="tight")
plt.close()

print("Done. Saved: vae_pytorch_generated_digits.png and vae_pytorch_latent_space.png")

Epoch  1/15 | loss: 192.4666 | recon: 188.5079 | kl: 3.9587
Epoch  2/15 | loss: 166.1159 | recon: 161.1764 | kl: 4.9395
Epoch  3/15 | loss: 161.0678 | recon: 155.8633 | kl: 5.2045
Epoch  4/15 | loss: 157.3093 | recon: 151.8553 | kl: 5.4539
Epoch  5/15 | loss: 154.5491 | recon: 148.8961 | kl: 5.6530
Epoch  6/15 | loss: 152.3957 | recon: 146.5839 | kl: 5.8119
Epoch  7/15 | loss: 150.6464 | recon: 144.7209 | kl: 5.9255
Epoch  8/15 | loss: 149.3237 | recon: 143.3032 | kl: 6.0205
Epoch  9/15 | loss: 148.1933 | recon: 142.1108 | kl: 6.0825
Epoch 10/15 | loss: 147.2903 | recon: 141.1356 | kl: 6.1547
Epoch 11/15 | loss: 146.4643 | recon: 140.2823 | kl: 6.1819
Epoch 12/15 | loss: 145.8302 | recon: 139.6124 | kl: 6.2178
Epoch 13/15 | loss: 145.2924 | recon: 139.0312 | kl: 6.2613
Epoch 14/15 | loss: 144.7075 | recon: 138.4096 | kl: 6.2979
Epoch 15/15 | loss: 144.2003 | recon: 137.8700 | kl: 6.3302
Done. Saved: vae_pytorch_generated_digits.png and vae_pytorch_latent_space.png
